# Phase 2 — Data Collection Notebook

This notebook reproduces the full data collection and merging pipeline originally implemented in `collect_data.py`.

Steps:
1. Download & prepare daily PM2.5 for NYC (EPA AQS pre-generated file)
2. Download NYC weather (Meteostat)
3. Download daily NYC traffic counts
4. Merge datasets into a single daily panel

**This notebook creates:**
- `data/raw/openaq_nyc_2023.csv`
- `data/raw/weather_nyc_2023.csv`
- `data/raw/traffic_nyc_2023.csv`
- `data/processed/nyc_air_weather_traffic_2023_daily.csv`

In [8]:
import os
import io
import zipfile
from datetime import datetime

import requests
import pandas as pd
from meteostat import Daily, Point

YEAR = 2023
EPA_PM25_ZIP_URL = f"https://aqs.epa.gov/aqsweb/airdata/daily_88101_{YEAR}.zip"
PM25_STATE_NAME = "New York"
PM25_CITY_NAME = "New York"

NYC_LAT, NYC_LON = 40.7128, -74.0060
NYC_TRAFFIC_CSV_URL = "https://data.cityofnewyork.us/resource/7ym2-wayt.csv"

## 1. Fetch & Prepare PM2.5 (EPA AQS Daily File)

In [9]:
def fetch_pm25_nyc():
    os.makedirs("../data/raw", exist_ok=True)
    print("Downloading EPA PM2.5...")

    resp = requests.get(EPA_PM25_ZIP_URL)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f)

    # Expected columns in EPA daily file
    required = ["State Name", "City Name", "Date Local", "Arithmetic Mean"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"Column {c} missing from EPA file. Columns: {df.columns.tolist()}")

    df_nyc = df[(df["State Name"] == PM25_STATE_NAME) & (df["City Name"] == PM25_CITY_NAME)].copy()
    df_nyc["Date Local"] = pd.to_datetime(df_nyc["Date Local"])
    df_nyc["date"] = df_nyc["Date Local"].dt.date

    daily = (
        df_nyc.groupby("date")["Arithmetic Mean"].mean().reset_index().rename(columns={"Arithmetic Mean": "pm25"})
    )

    daily = daily[(daily["pm25"] >= 0) & (daily["pm25"] < 500)]

    out = "../data/raw/openaq_nyc_2023.csv"
    daily.to_csv(out, index=False)
    print("Saved:", out)

fetch_pm25_nyc()

Saved: ../data/raw/openaq_nyc_2023.csv


## 2. Fetch Weather (Meteostat)

In [10]:
def fetch_weather_nyc():
    os.makedirs("../data/raw", exist_ok=True)
    print("Fetching Meteostat weather...")

    start = datetime(YEAR, 1, 1)
    end = datetime(YEAR, 12, 31)
    nyc = Point(NYC_LAT, NYC_LON)

    daily = Daily(nyc, start, end).fetch()
    df = daily.reset_index().rename(columns={"time": "date"})

    cols = df.columns
    out = pd.DataFrame()
    out["date"] = pd.to_datetime(df["date"]).dt.date

    out["temp_mean"] = df["tavg"] if "tavg" in cols else df[["tmin", "tmax"]].mean(axis=1)
    out["humidity_mean"] = df["rhum"] if "rhum" in cols else None
    out["wind_speed_mean"] = df["wspd"] if "wspd" in cols else None
    out["precip_mm"] = df["prcp"] if "prcp" in cols else 0.0

    out = out[(out["temp_mean"] > -50) & (out["temp_mean"] < 60)]

    out_path = "../data/raw/weather_nyc_2023.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)

fetch_weather_nyc()

Fetching Meteostat weather...
Saved: ../data/raw/weather_nyc_2023.csv


## 3. Fetch & Prepare NYC Traffic Data

In [11]:
def fetch_traffic_nyc():
    os.makedirs("../data/raw", exist_ok=True)
    print("Downloading NYC traffic...")

    params = {"$limit": 500000}
    resp = requests.get(NYC_TRAFFIC_CSV_URL, params=params)
    resp.raise_for_status()

    df = pd.read_csv(io.StringIO(resp.text))

    needed = ["yr", "m", "d", "vol"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing {c} in traffic dataset. Columns: {df.columns.tolist()}")

    df["yr"] = pd.to_numeric(df["yr"], errors="coerce")
    df["m"] = pd.to_numeric(df["m"], errors="coerce")
    df["d"] = pd.to_numeric(df["d"], errors="coerce")
    df["vol"] = pd.to_numeric(df["vol"], errors="coerce")

    df = df.dropna(subset=["yr", "m", "d", "vol"])
    df = df[df["yr"] == YEAR]

    df["date"] = pd.to_datetime(dict(year=df["yr"], month=df["m"], day=df["d"])).dt.date

    daily = df.groupby("date")["vol"].sum().reset_index().rename(columns={"vol": "traffic_volume"})
    daily = daily[daily["traffic_volume"] >= 0]

    out_path = "../data/raw/traffic_nyc_2023.csv"
    daily.to_csv(out_path, index=False)
    print("Saved:", out_path)

fetch_traffic_nyc()

Saved: ../data/raw/traffic_nyc_2023.csv


## 4. Merge All Datasets into Final Daily Panel

In [12]:
def merge_and_clean():
    os.makedirs("../data/processed", exist_ok=True)
    
    aq = pd.read_csv("../data/raw/openaq_nyc_2023.csv")
    weather = pd.read_csv("../data/raw/weather_nyc_2023.csv")
    traffic = pd.read_csv("../data/raw/traffic_nyc_2023.csv")

    aq["date"] = pd.to_datetime(aq["date"])
    weather["date"] = pd.to_datetime(weather["date"])
    traffic["date"] = pd.to_datetime(traffic["date"])

    df = aq.merge(weather, on="date", how="inner").merge(traffic, on="date", how="inner")

    df = df[(df["pm25"] >= 0) & (df["pm25"] < 500)]
    if "temp_mean" in df.columns:
        df = df[(df["temp_mean"] > -50) & (df["temp_mean"] < 60)]

    df["humidity_mean"] = pd.to_numeric(df["humidity_mean"], errors="coerce")
    df["traffic_volume"] = pd.to_numeric(df["traffic_volume"], errors="coerce")
    df = df[df["traffic_volume"] >= 0]

    df["weekday"] = df["date"].dt.weekday
    df["is_weekend"] = df["weekday"] >= 5

    median_traffic = df["traffic_volume"].median()
    df["traffic_level"] = (df["traffic_volume"] > median_traffic).map({False: "low", True: "high"})

    out_path = "../data/processed/nyc_air_weather_traffic_2023_daily.csv"
    df.to_csv(out_path, index=False)
    print("Saved final dataset:", out_path)

merge_and_clean()

Saved final dataset: ../data/processed/nyc_air_weather_traffic_2023_daily.csv
